# PersonaPlex Web UI (Google Colab)

This notebook installs PersonaPlex ("moshi-personaplex") and launches the Web UI.

## Prerequisites
- Enable GPU: Runtime → Change runtime type → **GPU**
- Accept the model license: https://huggingface.co/nvidia/personaplex-7b-v1
- You need a Hugging Face token (read-only is sufficient for downloads).


In [ ]:
# OS dependencies (for Opus / sphn)
!apt-get update -y
!apt-get install -y --no-install-recommends build-essential pkg-config libopus-dev


In [ ]:
# Clone the repo
# If you are running from your fork, set REPO_URL to your fork URL.

REPO_URL = "https://github.com/ozekimasaki/personaplex.git"
REPO_BRANCH = "main"
REPO_DIR = "personaplex"

# Always re-clone to avoid stale /content/personaplex from previous runs
!rm -rf {REPO_DIR}
!git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# Print the exact commit to make debugging reproducible
!git rev-parse --short HEAD

# Install moshi-personaplex WITHOUT dependencies to avoid sounddevice on Colab
!pip -q install -e moshi/. --no-deps

# Install the runtime deps we need for the server
!pip -q install -r colab/requirements-colab.txt

# Ensure torch exists and is reasonably recent. Colab usually provides a CUDA build already.
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())

In [ ]:
# Verify GPU availability and torch version constraints
import torch
from packaging import version

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDAが見つかりません。Colabで Runtime → Change runtime type → GPU を選び、Runtimeを再起動してから再実行してください。"
    )

torch_ver = version.parse(torch.__version__.split('+')[0])
if torch_ver < version.parse("2.2.0"):
    raise RuntimeError(
        f"torch={torch.__version__} は personaplex の要件 (torch>=2.2) 未満です。"
    )

print("torch OK:", torch.__version__, "cuda:", torch.cuda.get_device_name(0))

In [ ]:
# Check GPU VRAM (PersonaPlex 7B is heavy; T4 16GB is often insufficient)
import torch

total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
name = torch.cuda.get_device_name(0)
print(f"GPU: {name} / VRAM: {total_gb:.2f} GB")

# Heuristic: 7B-class full-duplex speech models can exceed ~16GB during initialization.
# If you hit OOM on T4, use a larger GPU (e.g., A100 40GB) or a quantized checkpoint if available.
if total_gb < 24:
    raise RuntimeError(
        "VRAMが不足している可能性が高いです。\n"
        f"検出: {name} / {total_gb:.2f} GB\n\n"
        "PersonaPlex 7B はT4(約16GB)だとロード時にOOMになりやすいです。\n"
        "対処:\n"
        "- Colabのより大きいGPU（例: A100 40GB）で実行\n"
        "- 量子化済みチェックポイント（4bit/8bit）が提供されている場合はそれを使う\n"
        "- どうしてもなら --device cpu（非常に遅い）\n"
    )

In [ ]:
# Hugging Face login (required to download the model weights)
from huggingface_hub import login
login()


In [ ]:
# Verify access to the gated Hugging Face repo (license acceptance + authentication)
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError

REPO_ID = "nvidia/personaplex-7b-v1"

try:
    # Small file first (also ensures token is actually used)
    _ = hf_hub_download(REPO_ID, "config.json")
    # This is required by moshi.server and is typically gated
    _ = hf_hub_download(REPO_ID, "voices.tgz")
    print("HF access OK for:", REPO_ID)
except GatedRepoError as e:
    raise RuntimeError(
        "Hugging Faceのモデルがgatedになっており、アクセスできません。\n"
        "1) まず https://huggingface.co/nvidia/personaplex-7b-v1 にアクセス\n"
        "2) ライセンス（Use/Agree等）を承諾\n"
        "3) このノートブックで login() を実行し直す\n"
        "4) その後このセルを再実行\n\n"
        f"詳細: {e}"
    )

In [ ]:
# Launch the server with a tunnel URL printed in the output.
# Stop this cell to shut down the server.

!python -m moshi.server --host 0.0.0.0 --port 8998 --gradio-tunnel